[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-3-ai-agents/adv-01-compounding-errors-and-loop-guardrails.ipynb)

# Advanced Discussion 1 — the arithmetic of autonomy

Compounding per-step errors, the value of validators, and a simulation of iteration caps and repeat guards.

**Optional advanced-discussion lab** for AI Agents and Agentic AI with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Simulation only; runs in seconds.

In [ ]:
import numpy as np
rng = np.random.RandomState(0)

## 1. errors compound: probability that an n-step task succeeds when every step must be right

In [ ]:
print("per-step reliability p -> success of an n-step task = p**n")
print("  p      n=5    n=10   n=20   n=50")
for p in (0.90, 0.95, 0.98, 0.99, 0.999):
    print("  %.3f  %.3f  %.3f  %.3f  %.3f" % (p, *[p ** n for n in (5, 10, 20, 50)]))
# how reliable must each step be for an 80% task success rate?
for n in (5, 10, 20, 50): print("  n=%2d steps: need p >= %.4f per step for 80%% task success" % (n, 0.8 ** (1 / n)))
# retries help only if a failure is DETECTED (a validator / test / schema check)
print("with detection: probability a step ends correct after up to r attempts, if a wrong attempt is caught with probability d")
p = 0.90
for d in (0.0, 0.5, 0.9, 1.0):
    for r in (1, 3):
        # each attempt: correct w.p. p; wrong-and-caught w.p. (1-p)*d -> retry; wrong-and-missed w.p. (1-p)*(1-d) -> accepted silently
        good = 0.0; carry = 1.0
        for _ in range(r):
            good += carry * p; carry *= (1 - p) * d
        print("  d=%.1f attempts=%d -> step correct %.4f   -> 10-step task %.3f" % (d, r, good, good ** 10))

## 2. a simulated agent loop (no LLM): steps needed, stochastic mistakes, guardrails

In [ ]:
def run(task_len, p_right, p_thrash, max_iters, repeat_guard, seed):
    """Each iteration the agent takes a step. A wrong step wastes an iteration; with probability p_thrash the agent then repeats
    the same wrong action 1-3 more times. A repeat guard stops the repetition after one extra try and forces a re-plan."""
    r = np.random.RandomState(seed); progress = iters = wasted = 0
    while progress < task_len and iters < max_iters:
        iters += 1
        if r.rand() < p_right: progress += 1; continue
        wasted += 1
        if r.rand() < p_thrash:
            extra = r.randint(1, 4)
            if repeat_guard: extra = min(extra, 1)
            iters += extra; wasted += extra
    return progress >= task_len, iters, wasted
print("\nsimulated 8-step task, per-step accuracy 0.85, thrash probability 0.5, 4000 runs each")
print("max_iters  repeat_guard   success   mean iterations   wasted iterations per run")
for mi in (8, 12, 20, 40):
    for guard in (False, True):
        res = [run(8, 0.85, 0.5, mi, guard, s) for s in range(4000)]
        print("%9d  %-12s  %.3f     %.2f              %.2f" % (mi, guard, np.mean([r[0] for r in res]), np.mean([r[1] for r in res]), np.mean([r[2] for r in res])))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.